In [1]:
import pandas as pd

In [2]:
test = pd.read_csv(r"Z:\processing\Output\Q_datagenerator_4_models.csv")
test.head()

,filepath,cnn_prediction,vgg16_prediction,resnet50_prediction,densenet121_prediction
0,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.002286,0.107392,1.715231e-28,1.035682e-06
1,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.003218,0.126113,1.930804e-22,3.785278e-07
2,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.003628,0.323335,1.564761e-26,8.840814e-06
3,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.004795,0.349127,1.702443e-01,2.779948e-06
4,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.003608,0.692855,3.135295e-22,1.323221e-06


In [3]:
opt_weights = [0.2429687500000001, 0.2488281250000001, 0.243359375, 0.2648437499999998]


In [5]:
test["ensemble_score"] = sum([x*y for x,y in zip([results['cnn_prediction'], 
                                                  results['vgg16_prediction'], 
                                                  results['resnet50_prediction'], 
                                                  results['densenet121_prediction']], opt_weights)])

In [6]:
test

,filepath,cnn_prediction,vgg16_prediction,resnet50_prediction,densenet121_prediction,ensemble_score
0,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.002286,0.107392,1.715231e-28,1.035682e-06,0.027278
1,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.003218,0.126113,1.930804e-22,3.785278e-07,0.032162
2,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.003628,0.323335,1.564761e-26,8.840814e-06,0.081339
3,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.004795,0.349127,1.702443e-01,2.779948e-06,0.129469
4,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.003608,0.692855,3.135295e-22,1.323221e-06,0.173279
...,...,...,...,...,...,...
495,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.001114,0.867565,4.852171e-29,2.166401e-05,0.216151
496,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.002558,0.806806,3.730687e-24,1.849282e-05,0.201382
497,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.001686,0.760644,7.737245e-33,5.637431e-05,0.189694
498,Z:\processing\Data\Extracted_Spectrogram_Full_...,0.000474,0.850229,1.482730e-33,3.364194e-05,0.211685


# Two Day Period of 304 Manifest

In [10]:
comparison_304_df = pd.read_csv(r"Z:\Processing\Data\spectrogram_manifest_to_score_dir\2022-11-27-1539_304_.csv", parse_dates=['recording_start_time_abs'])

In [15]:
comparison_304_df.shape

(3596225, 9)

In [21]:
short = comparison_304_df[(comparison_304_df['recording_start_time_abs'] >= pd.to_datetime("9/26/2020 15:30")) & 
                  (comparison_304_df['recording_start_time_abs'] <= pd.to_datetime("9/28/2020 15:40"))]

In [24]:
short['recording_start_time_abs'].max()

Timestamp('2020-09-28 15:30:02')

In [26]:
short.to_csv('304_2day_comparison.csv', index=False)

# Same two day period from 304 Inference with Ming Predict-Ensemble

In [27]:
predictensemble = pd.read_csv(r"Z:\deployment_304_prediction_full.csv", parse_dates=['recording_start_time_abs'])

In [28]:
short_pe = predictensemble[(predictensemble['recording_start_time_abs'] >= pd.to_datetime("9/26/2020 15:30")) & 
                  (predictensemble['recording_start_time_abs'] <= pd.to_datetime("9/28/2020 15:40"))]

In [31]:
short_pe.to_csv('304_2day_predictensemble.csv',index=False)

# Check CSV

In [33]:
import pandas as pd
import numpy as np
import glob
import os


In [38]:
current_dir = os.path.abspath("Z:\processing")
#current_dir = os.path.abspath("Y:\\NMML_CAEP_Acoustics\\DOS_CIBAS\\_Automated_Processing\\2022_retraining")
data_dir = os.path.join(current_dir, "Data")
spectrogram_manifest_to_score_dir = os.path.join(data_dir, "spectrogram_manifest_to_score_dir")

effort_dir = os.path.join(data_dir, "Effort")

In [34]:
path_to_file = r"Z:\processing\Data\spectrogram_manifest_to_score_dir\2023-02-14-1534_301.csv"
full_analysis_score_df = pd.read_csv(path_to_file)

C:\Users\mml\AppData\Local\Temp\ipykernel_11632\254443155.py:2: DtypeWarning: Columns (2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  full_analysis_score_df = pd.read_csv(path_to_file)


In [53]:
int(os.path.splitext(full_analysis_score_df['spectrogram_file_path'][0])[0].split("_")[-2:][0])

0

In [40]:
deployment = input('Which deployment are you processing?:\n').replace(" ","")
deployment = "".join(char for char in deployment if char.isdigit())

Which deployment are you processing?:
301


In [41]:
#%% get effort table

effort_table = pd.read_csv(os.path.join(effort_dir, "CIBA_3_Effort.csv")).set_index(('Deploy ID'))
#effort_table['Effort Start file (UTC)'] = effort_table['Effort Start file (UTC)'].astype('int64')
#effort_table['Effort End file (UTC)'] = effort_table['Effort End file (UTC)'].astype('int64')
current_deployment = effort_table.loc[int(deployment)]

audio_dir = current_deployment[6]#os.path.join(data_dir, "Raw_Audio_Full_Analysis")


effort_start = int("".join(["20", str(int(current_deployment[2]))]))
effort_end = int("".join(["20", str(int(current_deployment[4]))]))

effort_start = pd.to_datetime(effort_start, format=("%Y%m%d%H%M%S"))#.to_datetime64()
effort_end = pd.to_datetime(effort_end, format=("%Y%m%d%H%M%S"))#.to_datetime64()
print(current_deployment)

#%% functions
def get_filename(df, label, output_label_name, extension):
    df[output_label_name] = "".join([os.path.basename(df[label])])
    return df

def get_wave_time(df):
    df['recording_start_time_abs'] = "".join(["20", df['wave_filename'].split(".")[1]])
    return df

#%% Get waves and filter before generation
    
audio_filenames = glob.glob(audio_dir + '/*.wav')
print("Total number of New Audio Files to Generate Spectrograms:", len(audio_filenames))

audio_filename_df  = pd.DataFrame(audio_filenames, columns=(['filepath']))
print("constructing df")


#audio_filename_df = audio_filename_df.apply(get_filepath_datetime, axis=1)
audio_filename_df = audio_filename_df.apply(get_filename, axis=1, args = ('filepath', 'wave_filename', '.wav'))
#audio_filename_df['recording_start_time_abs'] = audio_filename_df['wave_filename'].split(

audio_filename_df = audio_filename_df.apply(get_wave_time, axis=1)
audio_filename_df['recording_start_time_abs'] = pd.to_datetime(audio_filename_df['recording_start_time_abs'], format=("%Y%m%d%H%M%S"))

audio_filename_df = audio_filename_df[(audio_filename_df['recording_start_time_abs']>= effort_start) & (audio_filename_df['recording_start_time_abs']<= effort_end)]


Location                                                             Possession W
Deploy date/time (AK local)                                        9/4/2020 17:18
Effort Start file (UTC)                                            200915000002.0
Recover date/time (AK local)                                                  NaN
Effort End file (UTC)                                              210225113002.0
comments                                                                      NaN
Path_to_waves                   Z:\Data\CIBA3 Overwinter 2020-2021\301D\604536840
Name: 301, dtype: object



KeyboardInterrupt



# get broken images

In [54]:
broken_301 = pd.read_csv(r"C:\Users\mml\zero_file_301.csv")
broken_305 = pd.read_csv(r"C:\Users\mml\zero_file_305.csv")

C:\Users\mml\AppData\Local\Temp\ipykernel_11632\218348221.py:1: DtypeWarning: Columns (2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  broken_301 = pd.read_csv(r"C:\Users\mml\zero_file_301.csv")


In [55]:
broken_301.columns


Index(['spectrogram_file_path', 'wave_filename', 'filepath',
       'recording_start_time_abs', 'time_wave_recording_start_abs',
       'time_spectrogram_start_relative', 'time_spectrogram_end_relative',
       'time_spectrogram_start_absolute', 'time_spectrogram_end_absolute',
       'file_size_kb'],
      dtype='object')

In [62]:
broken_301[(broken_301["file_size_kb"]<=50)].to_csv('broken_301.csv',index=False)